# Query understanding - large, multilabel classification

1. Map a query -> category. LLM style and Jev Style.
2. Then filter

## Boilerplate

Install deps, mount GDrive, prompt for your OpenAI Key (placed in your GDrive)

In [1]:
!pip install git+https://github.com/softwaredoug/cheat-at-search.git
!pip install typesafe-sdk
from cheat_at_search.data_dir import mount
mount(use_gdrive=True)
from cheat_at_search.search import run_strategy, graded_bm25, ndcgs, ndcg_delta, vs_ideal
from cheat_at_search.wands_data import corpus, judgments

corpus

  Cloning https://github.com/softwaredoug/cheat-at-search.git to /tmp/pip-req-build-915hici9
  Running command git clone --filter=blob:none --quiet https://github.com/softwaredoug/cheat-at-search.git /tmp/pip-req-build-915hici9
  Resolved https://github.com/softwaredoug/cheat-at-search.git to commit 51148d04a6c84d6bdf79d8af4a19960fcc3dc670
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 747.7/747.7 kB 6.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.7/3.7 MB 13.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 740.6/740.6 kB 43.0 MB/s eta 0:00:00
  Created wheel for cheat_at_search: filename=cheat_at_search-0.1.0-py3-none-any.whl size=1455533 sha256=4f610319f0c57244b7ae9af6b0785aa04d56e4a2966b4afe5bb2c6d6dee998ea
  Stored in directory: /tmp/pip-ephem-wheel-cache-5qf1xn3h/wheels/ce/b7/b5/5b6456ba5de93e3b9f0b94ddbeb32ac5a07345c3

,product_id,product_name,product_class,category hierarchy,product_description,product_features,rating_count,average_rating,review_count,features,doc_id,title,description,category,sub_category,cat_subcat,image_url,title_snowball,description_snowball
0,0,solid wood platform bed,Beds,Furniture / Bedroom Furniture / Beds & Headboa...,"good , deep sleep can be quite difficult to ha...",overallwidth-sidetoside:64.7|dsprimaryproducts...,15.0,4.5,15.0,"[overallwidth-sidetoside:64.7, dsprimaryproduc...",0,solid wood platform bed,"good , deep sleep can be quite difficult to ha...",Furniture,Bedroom Furniture,Furniture / Bedroom Furniture,https://storage.googleapis.com/product-ai-imag...,"Terms({'platform', 'solid', 'wood', 'bed'})","Terms({'like', 'rest', 'nice', 'this', 'other'..."
1,1,all-clad 7 qt . slow cooker,Slow Cookers,Kitchen & Tabletop / Small Kitchen Appliances ...,"create delicious slow-cooked meals , from tend...",capacityquarts:7|producttype : slow cooker|pro...,100.0,2.0,98.0,"[capacityquarts:7, producttype : slow cooker, ...",1,all-clad 7 qt . slow cooker,"create delicious slow-cooked meals , from tend...",Kitchen & Tabletop,Small Kitchen Appliances,Kitchen & Tabletop / Small Kitchen Appliances,https://storage.googleapis.com/product-ai-imag...,"Terms({'all', 'cooker', 'slow', '7', 'qt', 'cl...","Terms({'cook', 'not', 'morn', 'this', 'but', '..."
2,2,all-clad electrics 6.5 qt . slow cooker,Slow Cookers,Kitchen & Tabletop / Small Kitchen Appliances ...,prepare home-cooked meals on any schedule with...,features : keep warm setting|capacityquarts:6....,208.0,3.0,181.0,"[features : keep warm setting, capacityquarts:...",2,all-clad electrics 6.5 qt . slow cooker,prepare home-cooked meals on any schedule with...,Kitchen & Tabletop,Small Kitchen Appliances,Kitchen & Tabletop / Small Kitchen Appliances,https://storage.googleapis.com/product-ai-imag...,"Terms({'5', '6', 'cooker', 'all', 'slow', 'ele...","Terms({'cook', 'schedul', 'this', 'slow', 'mea..."
3,3,all-clad all professional tools pizza cutter,"Slicers, Peelers And Graters",Browse By Brand / All-Clad,this original stainless tool was designed to c...,overallwidth-sidetoside:3.5|warrantylength : l...,69.0,4.5,42.0,"[overallwidth-sidetoside:3.5, warrantylength :...",3,all-clad all professional tools pizza cutter,this original stainless tool was designed to c...,Browse By Brand,All-Clad,Browse By Brand / All-Clad,https://storage.googleapis.com/product-ai-imag...,"Terms({'pizza', 'profession', 'cutter', 'all',...","Terms({'to', 'through', 'was', 'origin', 'cook..."
4,4,baldwin prestige alcott passage knob with roun...,Door Knobs,Home Improvement / Doors & Door Hardware / Doo...,the hardware has a rich heritage of delivering...,compatibledoorthickness:1.375 '' |countryofori...,70.0,5.0,42.0,"[compatibledoorthickness:1.375 '' , countryofo...",4,baldwin prestige alcott passage knob with roun...,the hardware has a rich heritage of delivering...,Home Improvement,Doors & Door Hardware,Home Improvement / Doors & Door Hardware,https://storage.googleapis.com/product-ai-imag...,"Terms({'passag', 'baldwin', 'alcott', 'rosett'...","Terms({'atmospher', 'passag', 'to', 'rich', 'm..."
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
42989,42989,malibu pressure balanced diverter fixed shower...,Shower Panels,Home Improvement / Bathroom Remodel & Bathroom...,the malibu pressure balanced diverter fixed sh...,producttype : shower panel|spraypattern : rain...,3.0,4.5,2.0,"[producttype : shower panel, spraypattern : ra...",42989,malibu pressure balanced diverter fixed shower...,the malibu pressure balanced diverter fixed sh...,Home Improvement,Bathroom Remodel & Bathroom Fixtures,Home Improvement / Bathroom Remodel & Bathroom...,https://storage.googleapis.com/product-ai-imag...,"Terms({'pressur', 'head', 'balanc', 'shower', ...","Terms({'showerhead', 'into', 'spray', 'unit', ..."
42990,42990,emmeline 5 piece breakfast dining set,Dining Table Sets,Furniture / Kitchen 

## Query -> LIST of full classification

Previously we mapped a query to a single classification. Let's now generate a LIST of possible classifications.

**Notice diff from previous**

* Our pydantic type returns a LIST, not just a single FullyClassifiedClassifications

### Get top N classifications

In [2]:
corpus['category hierarchy'].value_counts()

,count
category hierarchy,
Furniture / Living Room Furniture / Chairs & Seating / Accent Chairs,1103
Furniture / Bedroom Furniture / Beds & Headboards / Beds,1051
Rugs / Area Rugs,985
Furniture / Living Room Furniture / Coffee Tables & End Tables / Coffee Tables,895
Furniture / Living Room Furniture / Coffee Tables & End Tables / End & Side Tables,757
...,...
Appliances / Kitchen Appliances / Ranges / All Ranges,1
Storage & Organization / Cleaning & Laundry Organization / Laundry Accessories,1
Home Improvement / Doors & Door Hardware / Door Hardware & Accessories / Door Hardware / Pull & Push Plate Door Hardware,1


### Construct structured outputs dynamically

In [3]:
from pydantic import BaseModel, Field, create_model
from typing import Literal


def literal_model(
    values: list[str],
    *,
    field_name: str = "value",
    model_name: str = "DynamicModel",
    model_description: str = "Select the best matching value.",
    field_description: str = "The best matching value.",
) -> type[BaseModel]:
    if not values:
        raise ValueError("values must not be empty")

    literal_type = list[Literal[*values]]

    return create_model(
        model_name,
        __doc__=model_description,
        **{
            field_name: (
                literal_type,
                Field(..., description=field_description),
            )
        },
    )

MAX_OPENAI_VALUES = 1000

has_category_corpus = corpus[(corpus['category hierarchy'].str.strip() != '')
                             & (~corpus['category hierarchy'].isna())]
has_category_corpus = has_category_corpus[~has_category_corpus['category hierarchy'].str.contains('"')]
legal_categories = has_category_corpus['category hierarchy'].value_counts().head(MAX_OPENAI_VALUES)
legal_categories = legal_categories.index.tolist()

ClassificationModel = literal_model(legal_categories,
                                    field_name="classifications",
                                    model_name="Classification",
                                    model_description=("A furniture / home-goods e-commerce search query "
                                                        "classified into one or more classifications"),
                                    field_description="The best categories matching the search query")
ClassificationModel

__main__.Classification

### Query classification convenience code

Below `AutoEnricher` from Cheat at Search remembers the settings, and then responds to each prompt one at a time.

In [4]:
from cheat_at_search.enrich import AutoEnricher
from cheat_at_search.data_dir import key_for_provider

# Setup an enrichment task
enricher = AutoEnricher(
     model="openai/gpt-5",
     system_prompt="You are a helpful furniture shopping agent that helps users construct search queries.",
     response_model=ClassificationModel
)

# Generate prompt for user query
def get_prompt_fully_qualified(query):
        prompt = f"""
        As a helpful agent, you'll recieve requests from users looking for furniture products.

        Return the best category. Or Unknown if none fits. Only choose a category when you're
        confident that one fits. Otherwise choose Unknown. Unknown can also be used if its ambiguous
        and more than one category seems to fit

        Here is the users request:

        {query}

        """

        return prompt

# Enrich the prompt
def fully_classified(query) -> str | None:
    prompt = get_prompt_fully_qualified(query)
    classification = enricher.enrich(prompt).model_copy()
    return classification.classifications

fully_classified("dinosaur"), fully_classified("sofa loveseat")

([], ['Furniture / Living Room Furniture / Sofas'])

### Do the same with Jev

We notice below, with Jev, we not only specify a list, but the creteria for each classification.

Unlike OpenAI, Jev comes with probabilities and confidence.

In [5]:
MAX_JEV_VALUES = 255
legal_jev_categories = has_category_corpus['category hierarchy'].value_counts().head(MAX_JEV_VALUES)

legal_jev_categories

,count
category hierarchy,
Furniture / Living Room Furniture / Chairs & Seating / Accent Chairs,1103
Furniture / Bedroom Furniture / Beds & Headboards / Beds,1051
Rugs / Area Rugs,985
Furniture / Living Room Furniture / Coffee Tables & End Tables / Coffee Tables,895
Furniture / Living Room Furniture / Coffee Tables & End Tables / End & Side Tables,757
...,...
Home Improvement / Bathroom Remodel & Bathroom Fixtures / Showers & Bathtubs / Shower & Bathtub Doors / Hinged Shower & Bathtub Doors,27
Outdoor / Garden / Planters / Plastic Planters,27
"Home Improvement / Flooring, Walls & Ceiling / Walls & Ceilings / Peel & Stick Backsplash Tile",26


In [6]:
from typesafe_sdk import TypeSafeClient, Choice

jev_key = key_for_provider('typesafe')
client = TypeSafeClient(api_key=jev_key)


classifications = {cat: None for cat in legal_jev_categories.index.tolist()}


resp = client.system_one(
    state="Search query: red couch for my office",
    questions={
        "category": Choice(
            instructions="Which product classification holds relevant products for this query?",
            criteria=classifications
        )
    }
)
resp

SystemOneResponse(model='jev-1.13.0', usage=Usage(input_tokens=6029, output_tokens=6022), answers={'category': ChoiceAnswer(type='choice', choice='Furniture / Living Room Furniture / Sofas', confidence=0.99, probabilities={'Baby & Kids / Toddler & Kids Bedroom Furniture / Kids Headboards': 0.0, 'Storage & Organization / Garage & Outdoor Storage & Organization / Deck Boxes & Patio Storage': 0.0, 'Baby & Kids / Toddler & Kids Bedroom Furniture / Kids Desks': 0.0, 'Home Improvement / Bathroom Remodel & Bathroom Fixtures / Bathroom Vanities / Vanity Bases': 0.0, 'Outdoor / Outdoor Décor / Statues & Sculptures / Animal Statues & Sculptures': 0.0, 'School Furniture and Supplies / School Furniture / School Tables / Folding Tables / Wood Folding Tables': 0.0, 'Décor & Pillows / Art / All Wall Art / Red Wall Art': 0.0, 'Kitchen & Tabletop / Tableware & Drinkware / Drinkware / All Drinkware': 0.0, 'Furniture / Kitchen & Dining Furniture / Dining Tables & Seating / Kitchen & Dining Tables / 6 Sea

In [7]:
resp.answers['category'].confidence

0.99

In [8]:
for classification, prob in resp.answers['category'].probabilities.items():
    if prob > 0:
        print(classification, prob)

Furniture / Living Room Furniture / Sofas 1.0


In [9]:
def jev_classifier(query: str, conf_threshold=0.7, prob_threshold=0.05) -> list[str] | None:
    """Classify a query with Jev"""
    resp = client.system_one(
        state=f"Search query: {query}",
        questions={
            "category": Choice(
                instructions="Which product category holds relevant products for this query?",
                criteria=classifications
            )
        }
    )
    category_answers = resp.answers['category']
    answers = []
    if category_answers.confidence > conf_threshold:
        probs = category_answers.probabilities
        for classification, prob in probs.items():
            if prob > prob_threshold:
                answers.append(classification)
    return answers

jev_classifier("bed")

['Furniture / Bedroom Furniture / Beds & Headboards / Beds']

## Run Category search strategy with classifier

Let's search, providing a boost when the product matches the category

In [13]:
from searcharray import SearchArray
from cheat_at_search.wands_data import corpus, judgments
from cheat_at_search.tokenizers import snowball_tokenizer
from cheat_at_search.strategy.strategy import SearchStrategy
import numpy as np


def parse_category(classification: str) -> str | None:
    toked = classification.split('/')
    if len(toked) > 1:
        return toked[0]
    return None

def parse_sub_category(classification: str) -> str | None:
    toked = classification.split('/')
    if len(toked) > 1:
        return '/'.join(toked[1:])
    return None

class CategorySearch(SearchStrategy):
    def __init__(self, products, query_to_cat,
                 name_boost=9.3,
                 description_boost=4.1,
                 category_boost=0,
                 sub_category_boost=0,
                 direct_boost=10):
        super().__init__(products, workers=16)
        self.index = products
        if 'title_snowball' not in self.index.columns:
            self.index['title_snowball'] = SearchArray.index(
                products['title'], snowball_tokenizer)
        if 'description_snowball' not in self.index.columns:
            self.index['description_snowball'] = SearchArray.index(
                products['description'], snowball_tokenizer)

        cat_split = products['category hierarchy'].fillna('').str.split("/")
        self.index['category_snowball'] = SearchArray.index(
            products['category'], snowball_tokenizer
        )
        self.index['sub_category_snowball'] = SearchArray.index(
            products['sub_category'], snowball_tokenizer
        )
        self.index['classification_snowball'] = SearchArray.index(
            products['category hierarchy'], snowball_tokenizer
        )

        self.query_to_cat = query_to_cat
        self.name_boost = name_boost
        self.description_boost = description_boost
        self.category_boost = category_boost
        self.sub_category_boost = sub_category_boost
        self.direct_boost = direct_boost

    def search(self, query, k=10):
        """Dumb baseline lexical search, but add a constant boost when
           the desired category or subcategory"""
        bm25_scores = np.zeros(len(self.index))
        classifications = self.query_to_cat(query)
        categories = [parse_category(classification) for classification in classifications]
        sub_categories = [parse_sub_category(classification) for classification in classifications]

        categories = set(categories)
        sub_categories = set(sub_categories)

        tokenized = snowball_tokenizer(query)

        # ****
        # Baseline BM25 search from before
        for token in tokenized:
            bm25_scores += self.index['title_snowball'].array.score(token) * self.name_boost
            bm25_scores += self.index['description_snowball'].array.score(
                token) * self.description_boost

        # ****
        # If there's a category, boost that by a constant amount
        for category in categories:
            tokenized_category = snowball_tokenizer(category)
            category_match = np.ones(len(self.index), dtype=np.int64)
            if tokenized_category:
                category_match = self.index['category_snowball'].array.score(tokenized_category) > 0
            bm25_scores[category_match] += self.category_boost

        for sub_category in sub_categories:
            tokenized_sub_category = snowball_tokenizer(sub_category)
            sub_category_match = np.ones(len(self.index), dtype=np.int64)
            if tokenized_sub_category:
                sub_category_match = self.index['category_snowball'].array.score(tokenized_sub_category) > 0
            bm25_scores[sub_category_match] += self.sub_category_boost

        # ***
        for classification in classifications:
            tokenized_classification = snowball_tokenizer(classification)
            classification_match = np.ones(len(self.index), dtype=np.int64)
            if tokenized_classification:
                classification_match = self.index['classification_snowball'].array.score(tokenized_classification) > 0
            bm25_scores[classification_match] += self.sub_category_boost

        top_k = np.argsort(-bm25_scores)[:k]
        scores = bm25_scores[top_k]

        return top_k, scores

gpt5_search = CategorySearch(corpus, fully_classified)
graded_gpt5 = run_strategy(gpt5_search, judgments)
graded_gpt5

2026-10-07 00:15:31,235 - searcharray.indexing - INFO - Indexing begins w/ 4 workers


INFO:searcharray.indexing:Indexing begins w/ 4 workers


2026-10-07 00:15:31,246 - searcharray.indexing - INFO - 0 Batch Start tokenization


INFO:searcharray.indexing:0 Batch Start tokenization


2026-10-07 00:15:31,249 - searcharray.indexing - INFO - Tokenizing 42994 documents


...

2026-10-07 00:15:36,288 - searcharray.indexing - INFO - (main thread) Processing 1 batch results


INFO:searcharray.indexing:(main thread) Processing 1 batch results


2026-10-07 00:15:36,323 - searcharray.indexing - INFO - Indexing from tokenization complete


INFO:searcharray.indexing:Indexing from tokenization complete
Searching: 100%|██████████| 480/480 [00:06<00:00, 79.21it/s] 


,product_id,product_name,product_class,category hierarchy,product_description,product_features,rating_count,average_rating,review_count,features,...,score,query,query_id,rank,grade,discounted_gain,idcg,dcg,ndcg,mrr
0,34988,magreb velvet upholstered parsons chair in olive,Dining Chairs,Furniture / Kitchen & Dining Furniture / Dinin...,this mid-century modern dining chair combines ...,dswoodtone : medium wood|supplierintendedandap...,1.0,5.0,1.0,"[dswoodtone : medium wood, supplierintendedand...",...,67.678887,oliver parsons,272,1,0.0,0.000000,8.786905,0.277778,0.031613,0.0
1,7732,olive branch wall décor,Wall Décor,Décor & Pillows / Wall Décor / Wall Accents,curving scrollwork branches topped with olives...,color : silver|dsmetallic : silver|primarymate...,107.0,4.5,74.0,"[color : silver, dsmetallic : silver, primarym...",...,56.360729,oliver parsons,272,2,0.0,0.000000,8.786905,0.277778,0.031613,0.0
2,22677,olive coffee table,Coffee & Cocktail Tables,Furniture / Living Room Furniture / Coffee Tab...,liven up your interiors with the super stylish...,topmaterial : manufactured wood|topcolor : wal...,9.0,4.0,7.0,"[topmaterial : manufactured wood, topcolor : w...",...,50.115616,oliver parsons,272,3,0.0,0.000000,8.786905,0.277778,0.031613,0.0
3,25622,soucy upholstered side chair in olive beige,Dining Chairs,Furniture / Kitchen & Dining Furniture / Dinin...,a contemporary take on a traditional dining ro...,dsprimaryproductstyle : boho|weightcapacity:25...,16.0,4.5,12.0,"[dsprimaryproductstyle : boho, weightcapacity:...",...,49.567551,oliver parsons,272,4,0.0,0.000000,8.786905,0.277778,0.031613,0.0
4,30646,oliver 40 '' wide power lift assist standard r...,Recliners,Furniture / Living Room Furniture / Chairs & S...,the oliver dual motor power lift recliner allo...,armheight-floortoarm:26|additionaltoolsrequire...,1.0,4.0,1.0,"[armheight-floortoarm:26, additionaltoolsrequi...",...,46.122177,oliver parsons,272,5,0.0,0.000000,8.786905,0.277778,0.031613,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4795,23533,albin height adjustable standing desk,Desks,Furniture / Office Furniture / Desks,instantly upgrade your workspace with this con...,topmaterial : manufactured wood|topcolor : whi...,245.0,4.5,171.0,"[topmaterial : manufactured wood, topcolor : w...",...,99.462148,gowdy height adjustable standing desk,274,6,1.0,0.166667,8.786905,4.928968,0.560945,1.0
4796,2969,electric height adjustable standing desk conve...,Commercial Desks|Desks,Furniture / Office Furniture / Desks,seamlessly convert your traditional desk into ...,keyboardtrayheight:5|overalldepth-fronttoback:...,68.0,4.5,39.0,"[keyboardtrayheight:5, overalldepth-fronttobac...",...,98.903410,gowdy height adjustable standing desk,274,7,1.0,0.142857,8.786905,4.928968,0.560945,1.0
4797,14515,lynnea height adjustable standing desk,NaN,Furniture / Office Furniture / Desks,featuring industrial aesthetic with a unique b...,dsprimaryproductstyle : country / farmhouse|wo...,205.0,4.5,144.0,"[dsprimaryproductstyle : country / farmhouse, ...",...,98.489351,gowdy height adjustable standing desk,274,8,1.0,0.125000,8.786905,4.928968,0.560945,1.0
4798,23570,kamen height adjustable standing desk converter,Commercial Desks|Desks,Furniture / Office Furniture / Desks,are you choosing to live a healthier lifestyle...,weightcapacity:160|overallproductweight:34|ove...,207.0,4.5,138.0,"[weightcapacity:160, overallproductweight:34, ...",...,97.792710,gowdy height adjustable standing desk,274,9,1.0,0.111111,8.786905,4.928968,0.560945,1.0


In [14]:
jev_search = CategorySearch(corpus, jev_classifier)
graded_jev = run_strategy(jev_search, judgments)
graded_jev

2026-10-07 00:15:42,640 - searcharray.indexing - INFO - Indexing begins w/ 4 workers


INFO:searcharray.indexing:Indexing begins w/ 4 workers


2026-10-07 00:15:42,648 - searcharray.indexing - INFO - 0 Batch Start tokenization


INFO:searcharray.indexing:0 Batch Start tokenization


2026-10-07 00:15:42,651 - searcharray.indexing - INFO - Tokenizing 42994 documents


...

2026-10-07 00:15:47,841 - searcharray.indexing - INFO - (main thread) Processing 1 batch results


INFO:searcharray.indexing:(main thread) Processing 1 batch results


2026-10-07 00:15:47,893 - searcharray.indexing - INFO - Indexing from tokenization complete


INFO:searcharray.indexing:Indexing from tokenization complete
Searching: 100%|██████████| 480/480 [00:10<00:00, 45.95it/s]


,product_id,product_name,product_class,category hierarchy,product_description,product_features,rating_count,average_rating,review_count,features,...,score,query,query_id,rank,grade,discounted_gain,idcg,dcg,ndcg,mrr
0,24366,brunk ship wheel sculpture,Decorative Objects,Décor & Pillows / Home Accessories / Decorativ...,adds a nautical flair to your home or business .,style : coastal|overallwidth-sidetoside:9|fant...,142.0,5.0,97.0,"[style : coastal, overallwidth-sidetoside:9, f...",...,112.732229,brunk ship wheel,281,1,2.0,3.000000,8.786905,4.928968,0.560945,1.0
1,18444,thetford ship wheel,Decorative Objects,Décor & Pillows / Home Accessories / Decorativ...,this 30 inch ship wheel is made from the fines...,animals : no animals|geometricshapes : no geom...,NaN,NaN,NaN,"[animals : no animals, geometricshapes : no ge...",...,88.601616,brunk ship wheel,281,2,1.0,0.500000,8.786905,4.928968,0.560945,1.0
2,630,ships wheel standup,Life Size Cutouts,Furniture / Game Tables & Game Room Furniture ...,this is a cardboard cutout of ships wheel stan...,countryoforigin : united states|lifestage : ki...,NaN,NaN,NaN,"[countryoforigin : united states, lifestage : ...",...,86.991166,brunk ship wheel,281,3,1.0,0.333333,8.786905,4.928968,0.560945,1.0
3,17326,6 '' decorative ship wheel,Wall Décor,Décor & Pillows / Wall Décor / Wall Accents,the decorative wooden ship wheel has six spoke...,overallheight-toptobottom:6|purposefuldistress...,7.0,4.0,4.0,"[overallheight-toptobottom:6, purposefuldistre...",...,85.397110,brunk ship wheel,281,4,1.0,0.250000,8.786905,4.928968,0.560945,1.0
4,18443,ship wheel wall décor,Wall Décor,Décor & Pillows / Wall Décor / Wall Accents,this ship wheel is made from the finest qualit...,individualpiecedepth-fronttoback:36|color : br...,1.0,4.0,1.0,"[individualpiecedepth-fronttoback:36, color : ...",...,84.711864,brunk ship wheel,281,5,1.0,0.200000,8.786905,4.928968,0.560945,1.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4795,42863,canadian maple leaf 16 oz . glass beer mug,Glassware & Barware,Kitchen & Tabletop / Tableware & Drinkware / D...,each classic beer mug in this set of four feat...,capacity:16|insulationtype : not insulated|set...,3.0,4.5,2.0,"[capacity:16, insulationtype : not insulated, ...",...,46.567793,canadian,335,6,2.0,0.500000,8.786905,8.786905,1.000000,1.0
4796,20641,58 '' hardwood canadian maple pool cue set,Game Table Accessories,Furniture / Game Tables & Game Room Furniture ...,made of hardwood canadian maple wood - steel j...,primarymaterial : wood|overallwidth-sidetoside...,32.0,5.0,21.0,"[primarymaterial : wood, overallwidth-sidetosi...",...,46.177668,canadian,335,7,2.0,0.428571,8.786905,8.786905,1.000000,1.0
4797,20643,58 '' hardwood canadian maple pool cue - billi...,Game Table Accessories,Furniture / Game Tables & Game Room Furniture ...,made of hardwood canadian maple wood - choose ...,warrantylength:30 days|overallproductweight:1....,19.0,4.5,15.0,"[warrantylength:30 days, overallproductweight:...",...,44.608887,canadian,335,8,2.0,0.375000,8.786905,8.786905,1.000000,1.0
4798,28028,canadian pine artificial christmas teardrop do...,Garlands & Swag,Décor & Pillows / Flowers & Plants / Garlands ...,this classical canadian pine artificial christ...,location : all outdoor use|plantspecies : pine...,1.0,5.0,1.0,"[location : all outdoor use, plantspecies : pi...",...,43.165700,canadian,335,9,2.0,0.333333,8.786905,8.786905,1.000000,1.0


### Analyze results

* We notice a small bump in performance in NDCG from GPT-5.
* We also notice a bigger bump with Jev

In [15]:
ndcgs(graded_bm25).mean(), ndcgs(graded_gpt5).mean(), ndcgs(graded_jev).mean()

(np.float64(0.5411098691836396),
 np.float64(0.5410743048969575),
 np.float64(0.5410743048969575))